##  Kiểm tra rò rỉ dữ liệu

Split chính thức của Big-Vul trên Hugging Face có dùng được không?

Trùng từng byte → trùng sau chuẩn hóa khoảng trắng
→ trùng nhóm thu thập → trùng cấu trúc sau khi xóa hết tên biến.


In [1]:
from datasets import load_dataset
import pandas as pd
import hashlib
import re

In [2]:
dsd = load_dataset("benjis/bigvul")
tr = dsd["train"].to_pandas()
te = dsd["test"].to_pandas()

print(f"Split chính thức:  train {len(tr):,}   test {len(te):,}")

Split chính thức:  train 150,908   test 33,050


### B1 — Trùng lặp chính xác

Có hàm nào xuất hiện **y hệt từng byte** ở cả train và test?

Dùng `set` để bỏ trùng trong từng phía, rồi lấy phần giao. Chia cho `test` chứ
không phải `train`, vì ta muốn biết *bao nhiêu phần trăm hàm bị trùng*.

In [3]:
s_tr = set(tr["func_before"])
s_te = set(te["func_before"])
dup = s_tr & s_te

print(f"Hàm riêng biệt : train {len(s_tr):,}   test {len(s_te):,}")
print(f"Trùng y hệt    : {len(dup):,}  ({len(dup)/len(s_te):.1%} của test)")

Hàm riêng biệt : train 150,908   test 31,710
Trùng y hệt    : 24,002  (75.7% của test)


### B2 — Trùng lặp gần

B1 bị lách dễ dàng: thêm một dấu cách là hai chuỗi khác nhau ngay. Nhưng với
trình biên dịch — và với mô hình — chúng là một hàm.

Nên gộp mọi chuỗi khoảng trắng thành một dấu cách rồi mới băm. Hai hàm chỉ khác
thụt dòng sẽ cho cùng một fingerprint.

In [4]:
def normalize_code(code):
    return re.sub(r"\s+", " ", str(code)).strip()


def code_fingerprint(code):
    return hashlib.sha256(normalize_code(code).encode("utf-8")).hexdigest()


demo = 'void  f( int   x )\n{\n\treturn x;\n}'
print(f"Gốc       : {demo!r}")
print(f"Chuẩn hóa : {normalize_code(demo)!r}")
print(f"Hash      : {code_fingerprint(demo)[:24]}...")


Gốc       : 'void  f( int   x )\n{\n\treturn x;\n}'
Chuẩn hóa : 'void f( int x ) { return x; }'
Hash      : 4d9e07d025f7ae05ea2c1873...


In [5]:
f_tr = {code_fingerprint(c) for c in tr["func_before"]}
f_te = {code_fingerprint(c) for c in te["func_before"]}
near = f_tr & f_te

print(f"Trùng sau chuẩn hóa : {len(near):,}  ({len(near)/len(f_te):.1%} của test)")
print(f"Phát hiện thêm so với trước chuẩn hoá  : {len(near) - len(dup):,}")

Trùng sau chuẩn hóa : 24,095  (76.1% của test)
Phát hiện thêm so với trước chuẩn hoá  : 93


In [6]:
for col in ["commit_id", "project", "CVE ID"]:
    a = set(tr[col].dropna())
    b = set(te[col].dropna())
    ov = a & b
    print(f"{col:11s}: {len(ov):>6,} / {len(b):>6,} nhóm của test cũng có ở train  ({len(ov)/len(b):6.1%})")    

commit_id  :  3,208 /  3,215 nhóm của test cũng có ở train  ( 99.8%)


project    :    289 /    289 nhóm của test cũng có ở train  (100.0%)


CVE ID     :  2,865 /  2,867 nhóm của test cũng có ở train  ( 99.9%)


### B4 — Clone cấu trúc

**Vấn đề**
```c
int add(int a, int b) { return a + b; }
int sum(int x, int y) { return x + y; }
```
Nếu gặp hash thì nó sẽ khác nhau về giá trị, tuy nhiên về ý nghĩa thì nó vẫn tương tự nhau.

In [7]:
C_KEYWORDS = {
    # C
    "auto", "break", "case", "char", "const", "continue", "default", "do",
    "double", "else", "enum", "extern", "float", "for", "goto", "if", "inline",
    "int", "long", "register", "restrict", "return", "short", "signed",
    "sizeof", "static", "struct", "switch", "typedef", "union", "unsigned",
    "void", "volatile", "while", "bool", "true", "false", "NULL",
    # C++
    "class", "public", "private", "protected", "virtual", "new", "delete",
    "namespace", "template", "typename", "this", "using", "try", "catch",
    "throw", "nullptr", "operator", "static_cast", "const_cast",
    "reinterpret_cast", "dynamic_cast", "friend", "explicit", "mutable",
}


RE_COMMENT = re.compile(r"//[^\n]*|/\*.*?\*/", re.S)
RE_STRING = re.compile(r'"(?:[^"\\]|\\.)*"')
RE_CHAR = re.compile(r"'(?:[^'\\]|\\.)*'")
RE_NUMBER = re.compile(r"\b\d+\b")
RE_IDENT = re.compile(r"[A-Za-z_]\w*")
RE_WS = re.compile(r"\s+")


def skeleton(code):
    s = RE_COMMENT.sub(" ", str(code))
    s = RE_STRING.sub('"S"', s)
    s = RE_CHAR.sub("'C'", s)          # <-- thêm dòng này
    s = RE_NUMBER.sub("0", s)
    s = RE_IDENT.sub(lambda m: m.group(0) if m.group(0) in C_KEYWORDS else "V", s)
    return RE_WS.sub(" ", s).strip()


def skeleton_hash(code):               # <-- cách 2 dòng trống (PEP 8)
    return hashlib.sha256(skeleton(code).encode("utf-8")).hexdigest()


print(skeleton('int add(int a, int b) { /* sum */ return a + b + 42; }'))


int V(int V, int V) { return V + V + 0; }


In [8]:
tr["_skel"] = [skeleton(c) for c in tr["func_before"]]
te["_skel"] = [skeleton(c) for c in te["func_before"]]

tr["_len"] = tr["_skel"].str.len()
te["_len"] = te["_skel"].str.len()

print(f"{'ngưỡng':<20} {'hàm test':>10} {'có clone cấu trúc trong train':>32}")
for min_len, label in [(0, "tất cả"), (200, "skeleton >= 200"), (500, "skeleton >= 500")]:
    t = tr[tr["_len"] >= min_len]
    e = te[te["_len"] >= min_len]
    train_hashes = {hashlib.sha256(x.encode()).hexdigest() for x in t["_skel"]}
    hit = pd.Series(
        [hashlib.sha256(x.encode()).hexdigest() in train_hashes for x in e["_skel"]],
        index=e.index,
    )
    print(f"{label:<20} {len(e):>10,} {f'{hit.sum():,} ({hit.mean():.1%})':>32}")

ngưỡng                 hàm test    có clone cấu trúc trong train


tất cả                   33,050                   26,562 (80.4%)


skeleton >= 200          12,002                    9,186 (76.5%)


skeleton >= 500           4,052                    3,093 (76.3%)


**Vì sao phải lọc theo độ dài skeleton?**


Nếu không lọc theo độ dài skeleton, các hàm ngắn có thể dẫn đến nhiều trường hợp trùng lặp không đáng kể, làm tăng tỷ lệ phát hiện clone cấu trúc trong train một cách giả tạo. Bằng cách đặt ngưỡng độ dài skeleton, chúng ta tập trung vào các hàm có cấu trúc phức tạp hơn, giảm thiểu nhiễu từ các hàm ngắn và cải thiện độ chính xác của phân tích.


**Kết quả cho thấy rằng:**


Nếu rò rỉ chỉ do hàm tầm thường, tỷ lệ tận 80%. Ngay cả với những hàm có 500 ký tự khung thuần cấu trúc, thì vẫn có 3 trên 4 hàm tìm thấy một bản sinh đôi cấu trúc chính xác đến từng token trong tập train. 


Hai hàm dài mà có khung giống hệt nhau đến mức đó thì không thể nào là trùng hợp ngẫu nhiên. Đó là copy-paste

In [9]:
e = te[te["_len"] >= 200]
t = tr[tr["_len"] >= 200]
train_hashes = {hashlib.sha256(x.encode()).hexdigest() for x in t["_skel"]}
hit = pd.Series(
    [hashlib.sha256(x.encode()).hexdigest() in train_hashes for x in e["_skel"]],
    index=e.index,
)

print("Trong số hàm test có skeleton >= 200 ký tự:")
for label in (0, 1):
    m = e["vul"] == label
    print(f"  vul={label}: {hit[m].sum():,} / {m.sum():,} có bản sao cấu trúc trong train ({hit[m].mean():.1%})")

Trong số hàm test có skeleton >= 200 ký tự:
  vul=0: 8,646 / 11,356 có bản sao cấu trúc trong train (76.1%)
  vul=1: 540 / 646 có bản sao cấu trúc trong train (83.6%)


**Rò rỉ nghiêng về phía nào?**

| | hàm test | có bản sao cấu trúc trong train |
|---|---|---|
| `vul=0` an toàn | 11,356 | 8,646 (76.1%) |
| `vul=1` có lỗ hổng | 646 | 540 (**83.6%**) |

Rò rỉ **nặng hơn ở lớp dương**. Mà precision, recall, F1 đều tính trên lớp
dương — nên rò rỉ thổi phồng đúng những con số sẽ được đem đi báo cáo. Đây là
trường hợp xấu nhất có thể xảy ra.

Vì sao nghiêng? Vì lớp dương chính là lớp bị commit phóng đại: một CVE vá 24 hàm
sinh ra nhiều bản gần trùng của cùng một kiểu lỗi. Lớp âm đa dạng hơn nhiều.

## Kết luận: split chính thức của Big-Vul không dùng được

| Test | Câu hỏi | Kết quả |
|---|---|---|
| B1 chính xác | Có hàm nào trùng từng byte giữa train và test? | có rò rỉ |
| B2 chuẩn hóa | ...bỏ qua thụt dòng và xuống dòng? | có rò rỉ |
| B3 **nhóm** | Đơn vị *thu thập* dữ liệu có bị xé đôi? | **99.8% commit của test cũng ở train** |
| B4 cấu trúc | ...bỏ qua tên biến và hằng số? | **76.5% thân hàm test có bản sinh đôi** |


### Nguyên nhân

Big-Vul tạo một dòng cho mỗi hàm mà commit vá CVE chạm tới. 217,007 dòng đến từ
khoảng 4,000 commit — trung vị 24 hàm/commit, cao nhất 1,692. Split chính thức
chia ngẫu nhiên **theo dòng**, nên một commit 24 hàm còn nguyên với xác suất
0.2^24 ≈ 1.7e-17. Thực tế là mọi commit đều bị xé đôi.

Mô hình không học "lỗ hổng trông như thế nào". Nó nhận ra code nó đã thấy rồi.